# Extra de S5 · MCP en notebooks · 0. Preparar el entorno

En este extra montas **dos servidores MCP** sin salir de notebooks:

1. **`mcp-datos-master`**: un LLM consulta la capa *master* del caso Retail, **en SQLite o en BigQuery** (dos conectores, la misma interfaz).
2. **`mcp-modelo-retencion`**: el modelo de retención va dentro del MCP; le pasas un cliente y devuelve la predicción.

Este notebook solo comprueba que tienes lo necesario. Tarda un minuto. Si algo sale en rojo, dice qué hacer.

In [1]:
import asyncio, json, os, sys, sqlite3
from pathlib import Path

AQUI = Path.cwd()
RAIZ = next(p for p in [AQUI, *AQUI.parents] if (p / '04-Proyecto-Base').is_dir())    # funciona en tu portátil y en un Codespace
PROYECTO = RAIZ / '04-Proyecto-Base'
BD = PROYECTO / 'data' / 'processed' / 'curso.db'
TRABAJO = AQUI / '_trabajo'
TRABAJO.mkdir(exist_ok=True)
print('Repositorio:', RAIZ.name, '| base de datos:', BD.name, 'existe' if BD.is_file() else 'NO EXISTE (mira el notebook 00)')

Repositorio: curso-ml-ia-desarrollo-produccion | base de datos: curso.db existe


## 1. Instalar lo que añade este extra

Solo hace falta una vez. Si ya hiciste el laboratorio de agentes de S5, `mcp` ya está.

In [2]:
%pip install -q -r requisitos-extra-mcp.txt
import importlib.metadata as md
for paquete in ('mcp', 'google-cloud-bigquery', 'pyarrow', 'mlflow', 'pandas'):
    print(f'{paquete:<24}', md.version(paquete))

Note: you may need to restart the kernel to use updated packages.
mcp                      1.30.0
google-cloud-bigquery    3.47.0
pyarrow                  26.0.1
mlflow                   3.18.0
pandas                   2.3.3


## 2. Los datos y el modelo del proyecto

Necesitas la base `curso.db` (capa master y variables) y el modelo registrado en MLflow. Los crea el pipeline de S1 (`python -m ia_mlops.pipeline`, unos 30 s con la muestra).

In [3]:
import sqlite3
problemas = []
if not BD.is_file():
    problemas.append('Falta curso.db: desde 04-Proyecto-Base ejecuta  python -m ia_mlops.pipeline')
else:
    con = sqlite3.connect(BD)
    for t in ('master_clientes', 'master_facturas', 'master_lineas', 'master_productos', 'fs_produccion'):
        try:
            print(f'{t:<18}', con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0], 'filas')
        except sqlite3.OperationalError:
            problemas.append(f'Falta la tabla {t}: ejecuta  python -m ia_mlops.explotacion  (crea fs_produccion) tras el pipeline')
    con.close()

sys.path.insert(0, str(PROYECTO))
from ia_mlops import alias_modelo as A
try:
    uri, mlc = A.conectar()
    print()
    print(A.texto_estado(A.estado(mlc)))
except A.ErrorAlias as e:
    problemas.append(str(e))

print()
print('TODO LISTO' if not problemas else 'FALTA:\n- ' + '\n- '.join(problemas))

master_clientes    150 filas
master_facturas    1430 filas
master_lineas      22117 filas
master_productos   3077 filas
fs_produccion      148 filas



Modelo registrado «retencion-90d»: 6 versiones
  v1   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v2   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v3   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v4   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v5   alias: -            valid_auc 0.6982 · test_auc 0.7471
  v6   alias: -            valid_auc 0.6982 · test_auc 0.7471
Alias: ninguno

TODO LISTO


## 3. BigQuery (opcional)

El notebook 01 tiene dos conectores. **SQLite funciona sin nada más.** Para el de BigQuery necesitas un proyecto de Google Cloud donde puedas crear un dataset:

```
gcloud auth application-default login
gcloud config set project TU_PROYECTO
export GCP_PROJECT=TU_PROYECTO        # en Windows (cmd): set GCP_PROJECT=TU_PROYECTO
```

Si no lo configuras, el notebook 01 se salta esas celdas y sigue con SQLite. **No uses el proyecto de tu trabajo**: el notebook crea y borra un dataset `curso_master` (22 000 filas, unos 2 MB; con el nivel gratuito de BigQuery no cuesta nada, pero es tu cuenta).

In [4]:
HAY_BQ = False
if os.environ.get('GCP_PROJECT'):
    try:
        import google.auth
        google.auth.default()
        HAY_BQ = True
    except Exception as e:                                      # noqa: BLE001
        print('GCP_PROJECT está definido, pero no hay credenciales:', str(e)[:140])
print('BigQuery:', f"listo en el proyecto {os.environ['GCP_PROJECT']}" if HAY_BQ else 'no configurado (el notebook 01 usará solo SQLite)')

BigQuery: no configurado (el notebook 01 usará solo SQLite)
